<a href="https://colab.research.google.com/github/Arshia-Moghbel/Processing-MRI-pictures-with-M.L./blob/main/brain_tumor_classifier_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MRI classifier: Google Colab runner

Run the cells in order. The notebook downloads a public, four-class MRI dataset, trains `Brain_Tumor_CNN.py`, then lets you upload a 2-D image for a class prediction. This is research/education software, not a clinical diagnostic tool.

Before running: in **Runtime → Change runtime type**, select **T4 GPU** if available. The download cell needs a free Kaggle account and API token; the token stays in the temporary Colab runtime.

In [ ]:
# Mount persistent storage, fetch/update this repository, and install its dependencies.
from google.colab import drive
from pathlib import Path
import subprocess
drive.mount('/content/drive')

PROJECT = Path('/content/mri-classifier')
REPOSITORY = 'https://github.com/Arshia-Moghbel/Processing-MRI-pictures-with-M.L..git'
if PROJECT.is_dir():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(PROJECT)], check=True)
%cd /content/mri-classifier
%pip -q install -r requirements.txt kagglehub

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Download the training dataset

This cell downloads Kaggle's **Brain Tumor MRI Dataset** by Masoud Nickparvar (four classes: glioma, meningioma, no tumor, pituitary). It uses the dataset's `Training` directory and creates a validation split during training. The reported validation score may be optimistic because this public dataset does not provide patient IDs.

Create a Kaggle API token at https://www.kaggle.com/settings/account and paste it only when Colab asks. You may instead save it as a Colab Secret named `KAGGLE_API_TOKEN`.

In [ ]:
import os
from pathlib import Path
import kagglehub

# kagglehub reads a KAGGLE_API_TOKEN Colab Secret automatically when available.
# Without a Secret, this opens a one-time token prompt in the Colab output.
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except Exception:
    pass

try:
    DOWNLOAD_ROOT = Path(kagglehub.dataset_download(
        'masoudnickparvar/brain-tumor-mri-dataset',
        output_dir='/content/brain-tumor-mri-data',
    ))
except Exception as error:
    raise RuntimeError(
        'Kaggle download failed. Add KAGGLE_API_TOKEN to Colab Secrets or run kagglehub.login(), then run this cell again.'
    ) from error

training_dirs = [
    item for item in DOWNLOAD_ROOT.rglob('*')
    if item.is_dir() and item.name.lower() == 'training'
]
if len(training_dirs) != 1:
    raise RuntimeError(f'Expected one Training directory under {DOWNLOAD_ROOT}, found: {training_dirs}')
DATASET_PATH = training_dirs[0]
classes = sorted(item.name for item in DATASET_PATH.iterdir() if item.is_dir())
if len(classes) < 2:
    raise RuntimeError(f'No class folders found in {DATASET_PATH}')
print('Dataset:', DATASET_PATH)
print('Classes:', classes)
print('Images per class:', {name: sum(1 for path in (DATASET_PATH / name).rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}) for name in classes})


## Configure one run

For `image` mode, use `mri-data/class_name/file.png`. For `nifti` mode, set `DATA_MODE` to `nifti` and put a `labels.csv` containing `path,label` under `mri-data`. If a patient has multiple samples, set `PATIENT_CSV` to `patient_ids.csv` so that patient cannot appear in both splits.

Start with `BATCH_SIZE = 4` and `FINE_TUNE_LAYERS = 0`. Do not use BraTS subject directory names as labels.

In [ ]:
from pathlib import Path

DATA_MODE = 'image'  # change only to 'nifti' when labels.csv exists
# The download cell sets DATASET_PATH. Replace it only to train on your own reviewed data.
DATASET_PATH = globals().get('DATASET_PATH', Path('/content/drive/MyDrive/mri-data'))
LABELS_CSV = DATASET_PATH / 'labels.csv'
PATIENT_CSV = None  # or DATASET_PATH / 'patient_ids.csv'
OUTPUT_DIR = Path('/content/drive/MyDrive/mri-artifacts/run-01')
IMAGE_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 12
VALIDATION_SIZE = 0.2
SEED = 42
WEIGHTS = 'imagenet'  # use 'none' only when ImageNet weights cannot be downloaded
LEARNING_RATE = 1e-4
FINE_TUNE_LAYERS = 0
FINE_TUNE_LEARNING_RATE = 1e-5

In [ ]:
# Train. This writes model and evaluation files to OUTPUT_DIR on Google Drive.
command = [
    'python', 'Brain_Tumor_CNN.py',
    '--data-mode', DATA_MODE,
    '--dataset-path', str(DATASET_PATH),
    '--output-dir', str(OUTPUT_DIR),
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--validation-size', str(VALIDATION_SIZE),
    '--seed', str(SEED),
    '--weights', WEIGHTS,
    '--learning-rate', str(LEARNING_RATE),
    '--fine-tune-layers', str(FINE_TUNE_LAYERS),
    '--fine-tune-learning-rate', str(FINE_TUNE_LEARNING_RATE),
]
if DATA_MODE == 'nifti':
    command += ['--labels-csv', str(LABELS_CSV)]
if PATIENT_CSV is not None:
    command += ['--patient-csv', str(PATIENT_CSV)]

import subprocess
subprocess.run(command, check=True)

## Review results and predict one 2-D image

Read `evaluation.json` before interpreting any result. It reports whether the split was at image or patient level. Inference below applies only to a model trained with `DATA_MODE = 'image'`; it outputs class scores, not a tumour location or clinical diagnosis.

In [ ]:
import json
print(json.dumps(json.loads((OUTPUT_DIR / 'evaluation.json').read_text()), indent=2))

# Upload one ordinary 2-D MRI image. The model must already have been trained in image mode.
from google.colab import files
from IPython.display import Image, display

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one PNG, JPG, JPEG, or BMP image.')
uploaded_name = next(iter(uploaded))
uploaded_path = Path('/content') / uploaded_name
if uploaded_path.suffix.lower() not in {'.png', '.jpg', '.jpeg', '.bmp'}:
    raise ValueError('Supported formats: PNG, JPG, JPEG, BMP.')

prediction_command = [
    'python', 'Brain_Tumor_CNN.py', '--predict-image', str(uploaded_path),
    '--model-path', str(OUTPUT_DIR / 'best_model.keras'),
    '--label-map', str(OUTPUT_DIR / 'label_map.json'),
]
completed = subprocess.run(prediction_command, check=True, text=True, capture_output=True)
prediction = json.loads(completed.stdout)
display(Image(filename=str(uploaded_path), width=300))
print(json.dumps(prediction, indent=2))